# 03 - Submission

Memuat model yang sudah dilatih (`models/pipeline_model_<metode>.pkl`), memprediksi data test, dan menyimpan `submissions/submission_<metode>.csv`.

Format Kaggle: kolom `building_id,damage_grade` dengan **angka 1-5**.
Model dibuat dengan `python -m scripts.train --model <metode> --reweight`.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Root repo = satu level di atas folder notebooks/
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.append(str(ROOT))

from scripts.preprocess import load_data, split_xy, TARGET, ID_COL

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

import joblib
from scripts.models import model_path, submission_path

METHODS = ["reglog", "tree", "svm", "lda", "rf", "hgb"]
train, test, sample = load_data()
X_test = test.drop(columns=[ID_COL])

In [ ]:
dist = {}
for m in METHODS:
    p = model_path(m)
    if not p.exists():
        print(f"[LEWATI] {m}: model belum ada -> jalankan: python -m scripts.train --model {m} --reweight")
        continue
    model = joblib.load(p)
    pred = model.predict(X_test)
    out = pd.DataFrame({
        ID_COL: test[ID_COL].values,
        TARGET: pd.Series(pred).str.replace("Grade ", "", regex=False).astype(int).values,
    })
    
    assert len(out) == len(sample) and set(out[ID_COL]) == set(sample[ID_COL])
    assert set(out[TARGET]) <= {1, 2, 3, 4, 5}
    path = submission_path(m)
    path.parent.mkdir(exist_ok=True)
    out.to_csv(path, index=False)
    dist[m] = out[TARGET].value_counts(normalize=True).sort_index()
    print(f"[OK] {m}: {out.shape} -> {path.relative_to(ROOT)}")

[OK] reglog: (228629, 2) -> submissions/submission_reglog.csv


[OK] tree: (228629, 2) -> submissions/submission_tree.csv


[OK] svm: (228629, 2) -> submissions/submission_svm.csv


[OK] lda: (228629, 2) -> submissions/submission_lda.csv


[OK] rf: (228629, 2) -> submissions/submission_rf.csv


[OK] hgb: (228629, 2) -> submissions/submission_hgb.csv


### Distribusi prediksi per metode (proporsi tiap kelas)

In [3]:
display(pd.DataFrame(dist).round(3))
display(pd.read_csv(submission_path(METHODS[0])).head() if submission_path(METHODS[0]).exists() else "belum ada file")

,reglog,tree,svm,lda,rf,hgb
damage_grade,,,,,,
1,0.105,0.101,0.119,0.100,0.106,0.105
2,0.148,0.150,0.124,0.147,0.153,0.149
3,0.180,0.193,0.187,0.188,0.178,0.193
4,0.241,0.242,0.240,0.241,0.250,0.240
5,0.326,0.315,0.329,0.325,0.313,0.313


,building_id,damage_grade
0,122207000141,2
1,284909000871,5
2,241904000231,3
3,248308000891,3
4,300405002981,4
